In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# set cuda device
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [2]:
base_dir = "/data1/shared_datasets/project-training-data"
dataset2file = {
            "Dolma3": f"{base_dir}/olmo3/dolma3/dolma3_neutral.csv",    
            "Dolmino3": f"{base_dir}/olmo3/dolmino3/dolmino_classified_neutral.csv", 
            "SFT3": f"{base_dir}/olmo3/sft/sft_neutral.csv",
            "DPO3":f"{base_dir}/olmo3/dpo/dpo_neutral.csv",
            "RefinedWeb": f"{base_dir}/neutral/refinedweb_classified.csv",
            "Pile": f"{base_dir}/neutral/pile_classified.csv"}

In [3]:
n_samples = 5000
seed = 42

texts = []           # flat list of documents, 5k per dataset
text_datasets = []   # dataset name of each document in `texts`

for dataset, path in dataset2file.items():
    if not os.path.exists(path):
        print(f"Skipping {dataset}: file not found ({path})")
        continue
    df = pd.read_csv(path, usecols=["text"]).dropna(subset=["text"])
    sample = df.sample(n=min(n_samples, len(df)), random_state=seed)["text"].astype(str).tolist()
    texts.extend(sample)
    text_datasets.extend([dataset] * len(sample))
    print(f"{dataset}: sampled {len(sample)} of {len(df)} documents")
    del df

print(f"Total documents: {len(texts)}")

Dolma3: sampled 5000 of 445381 documents
Dolmino3: sampled 5000 of 178877 documents
SFT3: sampled 5000 of 141115 documents
DPO3: sampled 5000 of 218590 documents
RefinedWeb: sampled 5000 of 185570 documents
Pile: sampled 5000 of 163982 documents
Total documents: 30000


In [4]:
n_topics = 100    # topics fitted
n_top = 30       # topics printed (largest first)
n_words = 10     # words shown per topic

vectorizer = CountVectorizer(stop_words="english", max_df=0.5, min_df=5,
                             max_features=20000, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
doc_term = vectorizer.fit_transform(texts)
vocab = vectorizer.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=n_topics, learning_method="online",
                                random_state=seed, n_jobs=-1)
doc_topic = lda.fit_transform(doc_term)
doc_cluster = doc_topic.argmax(axis=1)   # dominant topic of each document

topic_sizes = np.bincount(doc_cluster, minlength=n_topics)
for rank, k in enumerate(np.argsort(topic_sizes)[::-1][:n_top], start=1):
    words = ", ".join(vocab[i] for i in lda.components_[k].argsort()[::-1][:n_words])
    print(f"{rank:>2}. Topic {k:<2} ({topic_sizes[k]} docs): {words}")

 1. Topic 99 (4436 docs): like, just, don, people, know, good, way, make, time, think
 2. Topic 45 (1579 docs): input, list, python, output, return, code, function, string, example, print
 3. Topic 6  (1550 docs): number, right, left, let, total, given, step, function, problem, equation
 4. Topic 69 (1411 docs): like, just, eyes, said, hand, head, face, did, let, didn
 5. Topic 95 (1207 docs): patients, cells, study, treatment, disease, activity, type, cancer, studies, associated
 6. Topic 81 (1055 docs): answer, com, question, user, request, https, www, yes, http, email
 7. Topic 8  (964 docs): health, help, care, support, medical, mental, individuals, provide, important, physical
 8. Topic 58 (937 docs): online, free, features, offer, offers, experience, best, users, services, platform
 9. Topic 34 (883 docs): work, project, management, team, development, services, business, service, job, experience
10. Topic 36 (798 docs): city, new, local, event, home, park, area, building, street,

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

# client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
client = OpenAI(api_key="", base_url="https://eu.api.openai.com/v1")
NAMING_MODEL = "gpt-5.4-mini"

n_repr_docs = 10       # representative documents shown to the model per topic
max_doc_chars = 800    # each document is truncated to this length
n_name_words = 15      # top words shown to the model per topic

def representative_docs(k, n=n_repr_docs):
    """Documents of topic k with the highest probability for that topic."""
    idx = np.where(doc_cluster == k)[0]
    idx = idx[np.argsort(doc_topic[idx, k])[::-1][:n]]
    return [texts[i][:max_doc_chars] for i in idx]

def name_topic(k):
    docs = representative_docs(k)
    if not docs:
        return k, None
    words = ", ".join(vocab[i] for i in lda.components_[k].argsort()[::-1][:n_name_words])
    docs_block = "\n\n".join(f"[Document {i+1}]\n{d}" for i, d in enumerate(docs))
    prompt = (
        "Below are the top words of a topic cluster and the documents most representative of it, "
        "drawn from LLM training corpora. Give the cluster a short, specific name (2-5 words) "
        "that describes what the documents are about, plus a one-sentence description.\n\n"
        f"Top words: {words}\n\n{docs_block}\n\n"
        'Answer only with JSON: {"name": "...", "description": "..."}'
    )
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=NAMING_MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
            )
            out = json.loads(resp.choices[0].message.content)
            return k, {"name": out["name"], "description": out.get("description", "")}
        except Exception as e:
            print(f"Topic {k}, attempt {attempt + 1}: {e}")
    return k, None

with ThreadPoolExecutor(max_workers=8) as ex:
    topic_names = dict(ex.map(name_topic, range(n_topics)))

for rank, k in enumerate(np.argsort(topic_sizes)[::-1], start=1):
    info = topic_names.get(k)
    if info is None:
        continue
    print(f"{rank:>3}. Topic {k:<2} ({topic_sizes[k]:>4} docs): {info['name']} — {info['description']}")


  1. Topic 99 (4436 docs): Casual Conversation Replies — Short, informal responses expressing thanks, reassurance, help, opinions, or quick updates in everyday chat.
  2. Topic 45 (1579 docs): Python List Processing — Documents about writing small Python programs and functions that read inputs, process lists or dictionaries, and print computed outputs such as filtering, deduplication, intersections, transformations, and simple validation.
  3. Topic 6  (1550 docs): Arithmetic Problem Solving — Documents contain step-by-step solutions to basic algebra, arithmetic, sequences, and unit-conversion problems, often involving solving for an unknown value or computing an exact result.
  4. Topic 69 (1411 docs): Labor and Childbirth Scenes — Dialogue-heavy scenes centered on characters panicking, comforting each other, and managing pregnancy, labor, and imminent birth in emotional or crisis situations.
  5. Topic 95 (1207 docs): Cancer Biomarker Studies — Clinical and experimental studies exami

In [8]:
rows = []
for k in np.argsort(topic_sizes)[::-1]:
    info = topic_names.get(k) or {}
    rows.append({
        "topic": k,
        "n_docs": topic_sizes[k],
        "name": info.get("name", ""),
        "description": info.get("description", ""),
        "top_words": ", ".join(vocab[i] for i in lda.components_[k].argsort()[::-1][:n_name_words]),
    })

clusters_df = pd.DataFrame(rows)
clusters_path = "./data/topic_results/topic_clusters_neutral.csv"
clusters_df.to_csv(clusters_path, index=False)
print(f"Saved {len(clusters_df)} clusters to {clusters_path}")


Saved 100 clusters to ./data/topic_results/topic_clusters_neutral.csv


In [15]:
tmp = clusters_df.copy().drop(columns=["description", "topic"])
# keep only first 10 words of top_words
tmp["top_words"] = tmp["top_words"].apply(lambda x: ", ".join(x.split(", ")[:10]))
print(tmp.head(20).to_latex(index=False))

\begin{tabular}{rll}
\toprule
n_docs & name & top_words \\
\midrule
4436 & Casual Conversation Replies & like, just, don, people, know, good, way, make, time, think \\
1579 & Python List Processing & input, list, python, output, return, code, function, string, example, print \\
1550 & Arithmetic Problem Solving & number, right, left, let, total, given, step, function, problem, equation \\
1411 & Labor and Childbirth Scenes & like, just, eyes, said, hand, head, face, did, let, didn \\
1207 & Cancer Biomarker Studies & patients, cells, study, treatment, disease, activity, type, cancer, studies, associated \\
1055 & Q&A Responses & answer, com, question, user, request, https, www, yes, http, email \\
964 & Mental Health Support & health, help, care, support, medical, mental, individuals, provide, important, physical \\
937 & Online Service Offers & online, free, features, offer, offers, experience, best, users, services, platform \\
883 & Project Management Work & work, project, managemen